# 1. Cleaning Data

Input : `data/<keyword>_merge.csv` (hasil dari notebook crawling)
Output: `data/<keyword>_clean.csv` (hanya kolom `full_text` yang sudah bersih)

In [1]:
import re
import pandas as pd

KEYWORD = "karhutla"   
INPUT = f"data/{KEYWORD}_merge.csv"
OUTPUT = f"data/{KEYWORD}_clean.csv"

## Load data

In [2]:
data = pd.read_csv(INPUT)
print("Jumlah baris:", len(data))
data.head()

Jumlah baris: 280


,conversation_id_str,created_at,favorite_count,full_text,id_str,image_url,in_reply_to_screen_name,lang,location,quote_count,reply_count,retweet_count,tweet_url,user_id_str,username
0,2105522044175852020,2026-10-01T04:55:52.000Z,0,Penegakan hukum karhutla memang bergerak. Teta...,2105522060948856917,NaN,signal__decode,in,NaN,0,0,0,https://x.com/signal__decode/status/2105522060...,2084319198747234305,signal__decode
1,2105522044175852020,2026-10-01T04:55:48.000Z,0,Kejagung menerima 172 SPDP perkara pidana karh...,2105522044175852020,https://pbs.twimg.com/media/HThRI0SawAAe62S.jpg,NaN,in,NaN,0,1,0,https://x.com/signal__decode/status/2105522044...,2084319198747234305,signal__decode
2,2105521389986083115,2026-10-01T04:53:12.000Z,0,Polsek Menjalin Edukasi Warga Cegah Karhutla S...,2105521389986083115,https://pbs.twimg.com/media/HThQiFhbAAAen80.jpg,NaN,in,NaN,0,0,0,https://x.com/Polsek_Menjalin/status/210552138...,1916840631777083392,Polsek_Menjalin
3,2105521337599164741,2026-10-01T04:53:00.000Z,0,"*Karhutla di Amburawang Laut Meluas, Tim Gabun...",2105521337599164741,https://pbs.twimg.com/media/HThQfiiaUAAI7Wj.jpg,NaN,in,NaN,0,0,0,https://x.com/polsekmuarajawa/status/210552133...,1262290263872950273,polsekmuarajawa
4,2105521192312717428,2026-10-01T04:52:25.000Z,46,"Di tengah kepungan api dan asap, orangutan ber...",2105521192312717428,https://pbs.twimg.com/amplify_video_thumb/2105...,NaN,in,NaN,1,1,14,https://x.com/tvOneNews/status/210552119231271...,55507370,tvOneNews


## Fungsi cleansing
Langkahnya:
1. huruf kecil semua
2. hapus link, mention (@), hashtag (#), angka
3. hapus karakter khusus/emoji
4. rapikan spasi berlebih

In [8]:
def cleansing(sent):
    string = str(sent).lower()                     # huruf kecil
    string = re.sub(r'http\S+|www\.\S+', ' ', string)  # hapus link
    string = re.sub(r'@\w+', ' ', string)           # hapus mention
    string = re.sub(r'#\w+', ' ', string)           # hapus hashtag
    string = re.sub(r'\d+', ' ', string)            # hapus angka
    string = re.sub(r'[^\w\s]', ' ', string)        # hapus karakter khusus & emoji
    string = re.sub(r'_', ' ', string)              # underscore ikut \w, buang juga
    string = re.sub(r'\s+', ' ', string).strip()    # rapikan spasi
    return string

# coba dulu di satu contoh
cleansing("Asap makin tebal!!! @pemprov_sumsel cek https://t.co/abc #Karhutla 2026 😷")

'asap makin tebal cek'

In [4]:
data['full_text'] = data['full_text'].apply(cleansing)
data[['full_text']].head(10)

,full_text
0,penegakan hukum karhutla memang bergerak tetap...
1,kejagung menerima spdp perkara pidana karhutla...
2,polsek menjalin edukasi warga cegah karhutla s...
3,karhutla di amburawang laut meluas tim gabunga...
4,di tengah kepungan api dan asap orangutan beru...
5,cegah api sebelum berkobar polsek muara kaman ...
6,karhutla di samboja bakar hektare lahan gambut...
7,btw daerah karhutla udah hujan belum yaa semog...
8,terharu kementerian pertahanan jepang resmi me...
9,cegah karhutla polsek pangkalan banteng gencar...


## Ambil kolom teks saja + buang baris kosong/duplikat

In [5]:
data = data[['full_text']]
data = data[data['full_text'].str.len() > 0]          # buang teks kosong
data = data.drop_duplicates(subset='full_text')       # buang teks kembar (mis. retweet)
data = data.reset_index(drop=True)
print("Jumlah baris setelah cleaning:", len(data))
data.head()

Jumlah baris setelah cleaning: 261


,full_text
0,penegakan hukum karhutla memang bergerak tetap...
1,kejagung menerima spdp perkara pidana karhutla...
2,polsek menjalin edukasi warga cegah karhutla s...
3,karhutla di amburawang laut meluas tim gabunga...
4,di tengah kepungan api dan asap orangutan beru...


## Simpan ke folder data

In [6]:
data.to_csv(OUTPUT, index=False)
print("Tersimpan:", OUTPUT)

Tersimpan: data/karhutla_clean.csv
